In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Any, List, Optional
from scipy.signal import fftconvolve
from matplotlib.ticker import FormatStrFormatter


# ============================================================
# Shared GL / binomial weights
# ============================================================
def psi_sequence(alpha: float, T: int) -> np.ndarray:
    psi = np.empty(T + 1, dtype=float)
    psi[0] = 1.0
    for j in range(1, T + 1):
        psi[j] = psi[j - 1] * (j - 1 - alpha) / j
    return psi


def precompute_psi(alpha: float, T: int) -> np.ndarray:
    return psi_sequence(alpha, T)


# ============================================================
# Fractional differences
# ============================================================
def delta_alpha_prefix_1d(x_prefix: np.ndarray, alpha: float) -> np.ndarray:
    x_prefix = np.asarray(x_prefix, dtype=float).reshape(-1)
    T = x_prefix.shape[0] - 1
    psi = precompute_psi(alpha, T)
    return fftconvolve(x_prefix, psi, mode="full")[: T + 1]


def delta_alpha_prefix_rows(x_prefix: np.ndarray, alpha_vec: np.ndarray) -> np.ndarray:
    x_prefix = np.asarray(x_prefix, dtype=float)
    alpha_vec = np.asarray(alpha_vec, dtype=float).reshape(-1)
    T = x_prefix.shape[0] - 1
    n = x_prefix.shape[1]
    if alpha_vec.shape[0] != n:
        raise ValueError("alpha_vec must have one entry per coordinate.")

    y = np.zeros((T + 1, n), dtype=float)
    for i in range(n):
        y[:, i] = delta_alpha_prefix_1d(x_prefix[:, i], float(alpha_vec[i]))
    return y


# ============================================================
# Helpers for sampling A
# ============================================================
def random_orthogonal_matrix(n: int, rng: np.random.Generator) -> np.ndarray:
    M = rng.normal(size=(n, n))
    Q, R = np.linalg.qr(M)
    signs = np.sign(np.diag(R))
    signs[signs == 0] = 1.0
    Q = Q @ np.diag(signs)
    return Q


def sample_stable_A(
    n: int,
    rng: np.random.Generator,
    eig_low: float = -0.5,
    eig_high: float = 0.5,
    symmetric: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    eigvals = rng.uniform(eig_low, eig_high, size=n)

    if symmetric:
        Q = random_orthogonal_matrix(n, rng)
        A = Q @ np.diag(eigvals) @ Q.T
    else:
        while True:
            V = rng.normal(size=(n, n))
            if np.linalg.matrix_rank(V) == n and np.linalg.cond(V) < 20:
                break
        A = V @ np.diag(eigvals) @ np.linalg.inv(V)

    return A, eigvals


# ============================================================
# Simulator: general vector-alpha system
# ============================================================
def simulate_vector_fractional_system(
    A_star: np.ndarray,
    alpha_star_vec: np.ndarray,
    T: int,
    sigma_eta: float,
    rng: np.random.Generator,
    x0: Optional[np.ndarray] = None,
    x0_mean: float | np.ndarray = 0.0,
    x0_std: float = 1.0,
) -> np.ndarray:
    A_star = np.asarray(A_star, dtype=float)
    alpha_star_vec = np.asarray(alpha_star_vec, dtype=float).reshape(-1)
    n = A_star.shape[0]

    if A_star.shape != (n, n):
        raise ValueError("A_star must be square.")
    if alpha_star_vec.shape[0] != n:
        raise ValueError("alpha_star_vec must have length n.")

    x = np.zeros((T + 1, n), dtype=float)

    if x0 is None:
        x[0] = rng.normal(loc=x0_mean, scale=x0_std, size=n)
    else:
        x[0] = np.asarray(x0, dtype=float).reshape(n)

    psi_list = [precompute_psi(float(alpha_star_vec[i]), T) for i in range(n)]

    for t in range(T):
        eta = rng.normal(0.0, sigma_eta, size=n) if sigma_eta > 0 else np.zeros(n)
        mem = np.zeros(n, dtype=float)

        for i in range(n):
            for j in range(1, t + 2):
                mem[i] += psi_list[i][j] * x[t + 1 - j, i]

        x[t + 1] = A_star @ x[t] - mem + eta

    return x


# ============================================================
# Method 1: FO-GS
# ============================================================
def profiled_loss_and_rowhat_window(
    x_full: np.ndarray,
    row_idx: int,
    alpha: float,
    start: int,
    t_fit: int,
    ridge: float = 1e-8,
):
    """
    Row-wise profiled loss:
        y_i = [Delta^alpha x_{start+1}^{(i)}, ..., Delta^alpha x_{start+t_fit}^{(i)}]
        X   = [x_start, ..., x_{start+t_fit-1}]
    """
    assert start >= 0
    assert t_fit >= 1
    assert start + t_fit <= x_full.shape[0] - 1

    x_full = np.asarray(x_full, dtype=float)
    n = x_full.shape[1]

    x_row_prefix = x_full[: start + t_fit + 1, row_idx]
    y_row_full = delta_alpha_prefix_1d(x_row_prefix, alpha)
    y_row = y_row_full[start + 1 : start + t_fit + 1].reshape(1, t_fit)

    Xprev = x_full[start : start + t_fit].T
    G = Xprev @ Xprev.T + ridge * np.eye(n)
    rhs = y_row @ Xprev.T
    a_hat = np.linalg.lstsq(G, rhs.T, rcond=None)[0].T
    residual = y_row - a_hat @ Xprev
    loss = float(np.sum(residual * residual))

    return loss, a_hat.reshape(-1)


def build_alpha_grid(
    alpha_low: float,
    alpha_high: float,
    eps: Optional[float] = None,
    num: Optional[int] = None,
) -> np.ndarray:
    if eps is None and num is None:
        raise ValueError("At least one of eps or num must be provided.")
    if alpha_low > alpha_high:
        raise ValueError("alpha_low must be <= alpha_high.")

    if eps is not None:
        if eps <= 0:
            raise ValueError("eps must be positive.")
        grid = np.arange(alpha_low, alpha_high + 0.5 * eps, eps, dtype=float)
        if grid[-1] > alpha_high:
            grid[-1] = alpha_high
        if abs(grid[-1] - alpha_high) > 1e-12:
            grid = np.append(grid, alpha_high)
        return np.unique(grid)

    if num is None or num < 2:
        raise ValueError("num must be at least 2 when eps is not used.")
    return np.linspace(alpha_low, alpha_high, num, dtype=float)


def grid_search_alpha_row_window(
    x_full: np.ndarray,
    row_idx: int,
    start: int,
    t_fit: int,
    alpha_low: float,
    alpha_high: float,
    eps: Optional[float] = None,
    num: Optional[int] = None,
    ridge: float = 1e-8,
):
    alphas = build_alpha_grid(alpha_low=alpha_low, alpha_high=alpha_high, eps=eps, num=num)
    losses = np.zeros_like(alphas)

    best_loss = np.inf
    best_alpha = None
    best_row = None

    for k, alpha in enumerate(alphas):
        loss, a_hat = profiled_loss_and_rowhat_window(
            x_full=x_full,
            row_idx=row_idx,
            alpha=float(alpha),
            start=start,
            t_fit=t_fit,
            ridge=ridge,
        )
        losses[k] = loss

        if loss < best_loss:
            best_loss = loss
            best_alpha = float(alpha)
            best_row = a_hat

    return alphas, losses, best_alpha, best_row


def identify_vector_fractional_system(
    x_full: np.ndarray,
    start: int,
    t_fit: int,
    alpha_bounds: np.ndarray,
    eps_vec: Optional[np.ndarray] = None,
    num_vec: Optional[np.ndarray] = None,
    ridge: float = 1e-8,
):
    n = x_full.shape[1]
    alpha_bounds = np.asarray(alpha_bounds, dtype=float)
    if alpha_bounds.shape != (n, 2):
        raise ValueError(f"alpha_bounds must have shape ({n}, 2).")

    if eps_vec is not None:
        eps_vec = np.asarray(eps_vec, dtype=float).reshape(n)
    if num_vec is not None:
        num_vec = np.asarray(num_vec, dtype=int).reshape(n)

    alpha_hat = np.zeros(n, dtype=float)
    A_hat = np.zeros((n, n), dtype=float)
    details = []

    for i in range(n):
        eps_i = None if eps_vec is None else float(eps_vec[i])
        num_i = None if num_vec is None else int(num_vec[i])

        alphas, losses, best_alpha, best_row = grid_search_alpha_row_window(
            x_full=x_full,
            row_idx=i,
            start=start,
            t_fit=t_fit,
            alpha_low=float(alpha_bounds[i, 0]),
            alpha_high=float(alpha_bounds[i, 1]),
            eps=eps_i,
            num=num_i,
            ridge=ridge,
        )

        alpha_hat[i] = best_alpha
        A_hat[i, :] = best_row
        details.append(
            {
                "row_idx": i,
                "alpha_grid": alphas,
                "losses": losses,
                "alpha_hat": best_alpha,
                "a_hat": best_row.copy(),
                "best_loss": float(np.min(losses)),
            }
        )

    return {
        "alpha_hat": alpha_hat,
        "A_hat": A_hat,
        "details": details,
    }


def estimate_fosysid_single_trajectory(
    x: np.ndarray,
    alpha_bounds: np.ndarray,
    eps_vec: Optional[np.ndarray] = None,
    num_vec: Optional[np.ndarray] = None,
    ridge: float = 1e-8,
) -> Dict[str, Any]:
    """
    Method 1 with NO burn-in:
        start = 0, t_fit = T
    """
    x = np.asarray(x, dtype=float)
    T = x.shape[0] - 1
    return identify_vector_fractional_system(
        x_full=x,
        start=0,
        t_fit=T,
        alpha_bounds=alpha_bounds,
        eps_vec=eps_vec,
        num_vec=num_vec,
        ridge=ridge,
    )

# ============================================================
# Method 2: FO-BS
# ============================================================
@dataclass
class PaperBaselineResult:
    alpha_hat: np.ndarray
    A_hat_from_Atilde: np.ndarray
    A0_hat_from_Atilde: np.ndarray
    A_tilde_hat_ols: np.ndarray
    A_hat_from_row_search: np.ndarray
    A_tilde_structured_from_row_search: np.ndarray
    alpha_details: List[Dict[str, Any]]
    augmented_train_mse: float


def build_structured_A_tilde(A: np.ndarray, alpha: np.ndarray, p: int) -> np.ndarray:
    A = np.asarray(A, dtype=float)
    alpha = np.asarray(alpha, dtype=float).reshape(-1)
    n = A.shape[0]
    d = n * p
    A_tilde = np.zeros((d, d), dtype=float)

    A_tilde[:n, :n] = A + np.diag(alpha)
    for ell in range(1, p):
        D = np.diag([psi_sequence(float(alpha_i), ell + 1)[ell + 1] for alpha_i in alpha])
        A_tilde[:n, ell * n:(ell + 1) * n] = -D
    for ell in range(1, p):
        A_tilde[ell * n:(ell + 1) * n, (ell - 1) * n:ell * n] = np.eye(n)
    return A_tilde


def build_augmented_states(x: np.ndarray, p: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    T = x.shape[0] - 1
    n = x.shape[1]
    xt = np.zeros((T + 1, n * p), dtype=float)
    for k in range(T + 1):
        xt[k] = np.concatenate([x[k - lag] if k - lag >= 0 else np.zeros(n) for lag in range(p)])
    return xt


def fit_row_structured_given_alpha(x: np.ndarray,
                                   i: int,
                                   alpha_i: float,
                                   p: int,
                                   ridge: float = 1e-8) -> Dict[str, Any]:
    r"""
    For candidate alpha_i, estimate row_i(A) by OLS under
        x_i[k+1] = row_i(A) x[k]
                   - alpha_i x_i[k]
                   - sum_{ell=1}^{p-1} psi(alpha_i, ell+1) x_i[k-ell]
                   + noise.
    """
    x = np.asarray(x, dtype=float)
    T = x.shape[0] - 1
    n = x.shape[1]
    if not (0 <= i < n):
        raise ValueError("row index i out of range.")
    if p < 1:
        raise ValueError("p must be >= 1.")

    psi = psi_sequence(float(alpha_i), p + 1)
    X = x[:T]
    y = x[1:, i].copy()

    y_adj = y + alpha_i * x[:T, i]
    for ell in range(1, p):
        coef = psi[ell + 1]
        for k in range(T):
            idx = k - ell
            if idx >= 0:
                y_adj[k] += coef * x[idx, i]

    G = X.T @ X + ridge * np.eye(n)
    rhs = X.T @ y_adj
    a_row = np.linalg.lstsq(G, rhs.T, rcond=None)[0].T

    row_tilde = np.zeros(n * p, dtype=float)
    row_tilde[:n] = a_row
    row_tilde[i] += alpha_i
    for ell in range(1, p):
        row_tilde[ell * n + i] = -psi[ell + 1]

    residual = y_adj - X @ a_row
    return {
        "alpha_i": float(alpha_i),
        "a_row": a_row,
        "row_tilde": row_tilde,
        "train_mse": float(np.mean(residual ** 2)),
    }


def rollout_coordinate_with_structured_row(x: np.ndarray,
                                           i: int,
                                           a_row: np.ndarray,
                                           alpha_i: float,
                                           p: int) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    T = x.shape[0] - 1
    psi = psi_sequence(float(alpha_i), p + 1)

    xhat = np.zeros(T + 1, dtype=float)
    xhat[0] = x[0, i]
    for k in range(T):
        mem = alpha_i * xhat[k]
        for ell in range(1, p):
            idx = k - ell
            if idx >= 0:
                mem += psi[ell + 1] * xhat[idx]
        xhat[k + 1] = float(a_row @ x[k] - mem)
    return xhat


def score_candidate_alpha(x: np.ndarray,
                          i: int,
                          alpha_i: float,
                          p: int,
                          ridge: float = 1e-8,
                          mse_mode: str = "rollout") -> Dict[str, Any]:
    fit = fit_row_structured_given_alpha(x=x, i=i, alpha_i=alpha_i, p=p, ridge=ridge)
    a_row = fit["a_row"]

    if mse_mode == "rollout":
        pred = rollout_coordinate_with_structured_row(x=x, i=i, a_row=a_row, alpha_i=alpha_i, p=p)
    elif mse_mode == "one_step":
        T = x.shape[0] - 1
        pred = np.zeros(T + 1, dtype=float)
        pred[0] = x[0, i]
        psi = psi_sequence(float(alpha_i), p + 1)
        for k in range(T):
            mem = alpha_i * x[k, i]
            for ell in range(1, p):
                idx = k - ell
                if idx >= 0:
                    mem += psi[ell + 1] * x[idx, i]
            pred[k + 1] = float(a_row @ x[k] - mem)
    else:
        raise ValueError("mse_mode must be 'rollout' or 'one_step'.")

    fit["prediction"] = pred
    fit["mse"] = float(np.mean((pred - x[:, i]) ** 2))
    return fit


def estimate_alpha_i_paper_bisection(x: np.ndarray,
                                     i: int,
                                     p: int,
                                     tol: float = 1e-2,
                                     max_iter: int = 25,
                                     ridge: float = 1e-8,
                                     alpha_left: float = 0.05,
                                     alpha_right: float = 0.95,
                                     mse_mode: str = "rollout",
                                     update_rule: str = "literal_endpoints") -> Dict[str, Any]:
    left = float(alpha_left)
    right = float(alpha_right)
    history: List[Dict[str, Any]] = []

    for it in range(max_iter):
        center = 0.5 * (left + right)
        cand_left = score_candidate_alpha(x=x, i=i, alpha_i=left, p=p, ridge=ridge, mse_mode=mse_mode)
        cand_center = score_candidate_alpha(x=x, i=i, alpha_i=center, p=p, ridge=ridge, mse_mode=mse_mode)
        cand_right = score_candidate_alpha(x=x, i=i, alpha_i=right, p=p, ridge=ridge, mse_mode=mse_mode)

        history.append({
            "iter": it,
            "left": left,
            "center": center,
            "right": right,
            "mse_left": cand_left["mse"],
            "mse_center": cand_center["mse"],
            "mse_right": cand_right["mse"],
        })

        if (right - left) < tol:
            break

        if update_rule == "literal_endpoints":
            if cand_left["mse"] <= cand_right["mse"]:
                right = center
            else:
                left = center
        elif update_rule == "practical_midpoint":
            mses = np.array([cand_left["mse"], cand_center["mse"], cand_right["mse"]])
            best = int(np.argmin(mses))
            if best == 0:
                right = center
            elif best == 2:
                left = center
            else:
                half = 0.25 * (right - left)
                left = center - half
                right = center + half
        else:
            raise ValueError("update_rule must be 'literal_endpoints' or 'practical_midpoint'.")

    alpha_hat = 0.5 * (left + right)
    final_fit = score_candidate_alpha(x=x, i=i, alpha_i=alpha_hat, p=p, ridge=ridge, mse_mode=mse_mode)
    return {
        "i": i,
        "alpha_hat": float(alpha_hat),
        "a_row_hat": final_fit["a_row"],
        "row_tilde_hat": final_fit["row_tilde"],
        "final_mse": final_fit["mse"],
        "history": history,
    }


def estimate_alpha_vector_paper_bisection(x: np.ndarray,
                                          p: int,
                                          tol: float = 1e-2,
                                          max_iter: int = 25,
                                          ridge: float = 1e-8,
                                          alpha_left: float = 0.05,
                                          alpha_right: float = 0.95,
                                          mse_mode: str = "rollout",
                                          update_rule: str = "literal_endpoints") -> Dict[str, Any]:
    x = np.asarray(x, dtype=float)
    n = x.shape[1]
    alpha_hat = np.zeros(n, dtype=float)
    A_from_row_search = np.zeros((n, n), dtype=float)
    details = []

    for i in range(n):
        res_i = estimate_alpha_i_paper_bisection(
            x=x,
            i=i,
            p=p,
            tol=tol,
            max_iter=max_iter,
            ridge=ridge,
            alpha_left=alpha_left,
            alpha_right=alpha_right,
            mse_mode=mse_mode,
            update_rule=update_rule,
        )
        alpha_hat[i] = res_i["alpha_hat"]
        A_from_row_search[i] = res_i["a_row_hat"]
        details.append(res_i)

    A_tilde_structured = build_structured_A_tilde(A_from_row_search, alpha_hat, p)
    return {
        "alpha_hat": alpha_hat,
        "A_from_row_search": A_from_row_search,
        "A_tilde_structured": A_tilde_structured,
        "details": details,
    }


def fit_augmented_ols_A_tilde(x: np.ndarray, p: int, ridge: float = 1e-8) -> Dict[str, Any]:
    xt = build_augmented_states(x, p=p)
    X = xt[:-1].T
    Y = xt[1:].T
    d = X.shape[0]
    G = X @ X.T + ridge * np.eye(d)
    RHS = Y @ X.T
    A_tilde_hat = np.linalg.solve(G.T, RHS.T).T
    residual = Y - A_tilde_hat @ X
    return {
        "A_tilde_hat": A_tilde_hat,
        "augmented_train_mse": float(np.mean(residual ** 2)),
        "xtilde": xt,
    }


def recover_A_from_Atilde(A_tilde_hat: np.ndarray,
                          alpha_hat: np.ndarray,
                          mode: str = "user_minus") -> Dict[str, Any]:
    alpha_hat = np.asarray(alpha_hat, dtype=float).reshape(-1)
    n = alpha_hat.shape[0]
    A0_hat = np.asarray(A_tilde_hat[:n, :n], dtype=float)
    if mode == "user_minus":
        A_hat = A0_hat - np.diag(alpha_hat)
    elif mode == "paper_plus":
        A_hat = A0_hat + np.diag(alpha_hat)
    else:
        raise ValueError("mode must be 'user_minus' or 'paper_plus'.")
    return {"A0_hat": A0_hat, "A_hat": A_hat}


def estimate_dt_fods_paper_aligned(x: np.ndarray,
                                   p: int,
                                   tol: float = 1e-2,
                                   max_iter: int = 25,
                                   ridge: float = 1e-8,
                                   alpha_left: float = 0.05,
                                   alpha_right: float = 0.95,
                                   mse_mode: str = "rollout",
                                   update_rule: str = "literal_endpoints",
                                   recover_A_mode: str = "user_minus") -> PaperBaselineResult:
    alpha_step = estimate_alpha_vector_paper_bisection(
        x=x,
        p=p,
        tol=tol,
        max_iter=max_iter,
        ridge=ridge,
        alpha_left=alpha_left,
        alpha_right=alpha_right,
        mse_mode=mse_mode,
        update_rule=update_rule,
    )
    alpha_hat = alpha_step["alpha_hat"]

    ols_step = fit_augmented_ols_A_tilde(x=x, p=p, ridge=ridge)
    rec_ols = recover_A_from_Atilde(
        A_tilde_hat=ols_step["A_tilde_hat"],
        alpha_hat=alpha_hat,
        mode=recover_A_mode,
    )

    rec_structured = recover_A_from_Atilde(
        A_tilde_hat=alpha_step["A_tilde_structured"],
        alpha_hat=alpha_hat,
        mode=recover_A_mode,
    )

    return PaperBaselineResult(
        alpha_hat=alpha_hat,
        A_hat_from_Atilde=rec_ols["A_hat"],
        A0_hat_from_Atilde=rec_ols["A0_hat"],
        A_tilde_hat_ols=ols_step["A_tilde_hat"],
        A_hat_from_row_search=rec_structured["A_hat"],
        A_tilde_structured_from_row_search=alpha_step["A_tilde_structured"],
        alpha_details=alpha_step["details"],
        augmented_train_mse=ols_step["augmented_train_mse"],
    )




# ============================================================
# FO-WT
# ============================================================
def haar_dwt_multilevel(signal: np.ndarray, max_level: Optional[int] = None) -> List[np.ndarray]:
    a = np.asarray(signal, dtype=float).reshape(-1)
    if a.size < 2:
        return []

    Jmax = int(np.floor(np.log2(a.size)))
    if max_level is None:
        max_level = Jmax
    max_level = int(max(0, min(max_level, Jmax)))

    details: List[np.ndarray] = []
    current = a.copy()
    for _ in range(max_level):
        m = (current.size // 2) * 2
        if m < 2:
            break
        c = current[:m]
        even = c[0::2]
        odd = c[1::2]
        detail = (even - odd) / np.sqrt(2.0)
        approx = (even + odd) / np.sqrt(2.0)
        details.append(detail)
        current = approx
    return details


def _linear_detrend_1d(x: np.ndarray) -> np.ndarray:
    t = np.arange(x.size, dtype=float)
    X = np.column_stack([np.ones_like(t), t])
    beta, *_ = np.linalg.lstsq(X, x, rcond=None)
    return x - X @ beta


def weighted_line_fit(x: np.ndarray, y: np.ndarray, w: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    X = np.column_stack([np.ones_like(x), x])
    sqrt_w = np.sqrt(np.clip(w, 1e-12, None))
    Xw = X * sqrt_w[:, None]
    yw = y * sqrt_w
    beta, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
    intercept, slope = beta
    return {"intercept": float(intercept), "slope": float(slope)}


def slope_to_alpha_direct(
    slope: float,
    alpha_min: float = 0.05,
    alpha_max: float = 0.95,
) -> float:
    return float(np.clip(0.5 * slope, alpha_min, alpha_max))


@dataclass
class WaveletAlpha1DResult:
    alpha_hat: float
    slope_hat: float
    intercept_hat: float
    levels_used: np.ndarray
    log2_vars_used: np.ndarray
    weights_used: np.ndarray
    all_levels: np.ndarray
    all_log2_vars: np.ndarray
    all_counts: np.ndarray


def estimate_alpha_wavelet_1d(
    series: np.ndarray,
    alpha_min: float = 0.05,
    alpha_max: float = 0.95,
    min_level: int = 2,
    max_level: Optional[int] = None,
    detrend: str = "linear",
) -> WaveletAlpha1DResult:
    x = np.asarray(series, dtype=float).reshape(-1)
    if x.size < 8:
        raise ValueError("Need at least 8 samples for the wavelet baseline.")

    if detrend == "linear":
        z = _linear_detrend_1d(x)
    elif detrend == "mean":
        z = x - np.mean(x)
    elif detrend == "none":
        z = x.copy()
    else:
        raise ValueError("detrend must be one of {'linear', 'mean', 'none'}.")

    details = haar_dwt_multilevel(z, max_level=max_level)
    if len(details) < 2:
        raise ValueError("Not enough wavelet levels. Increase T or lower max_level.")

    levels = []
    log2_vars = []
    counts = []
    eps = 1e-18

    for lev, d in enumerate(details, start=1):
        if d.size < 2:
            continue
        var = np.var(d, ddof=1)
        levels.append(lev)
        log2_vars.append(np.log2(max(var, eps)))
        counts.append(d.size)

    levels = np.asarray(levels, dtype=float)
    log2_vars = np.asarray(log2_vars, dtype=float)
    counts = np.asarray(counts, dtype=float)

    if levels.size < 2:
        raise ValueError("Not enough valid wavelet levels after variance computation.")

    if max_level is None:
        max_level_use = int(levels[-1] - 1) if levels.size >= 4 else int(levels[-1])
    else:
        max_level_use = int(max_level)

    mask = (levels >= float(min_level)) & (levels <= float(max_level_use))
    if np.sum(mask) < 2:
        mask = np.ones_like(levels, dtype=bool)

    fit = weighted_line_fit(levels[mask], log2_vars[mask], counts[mask])
    slope = fit["slope"]
    intercept = fit["intercept"]
    alpha_hat = slope_to_alpha_direct(slope, alpha_min=alpha_min, alpha_max=alpha_max)

    return WaveletAlpha1DResult(
        alpha_hat=float(alpha_hat),
        slope_hat=float(slope),
        intercept_hat=float(intercept),
        levels_used=levels[mask].copy(),
        log2_vars_used=log2_vars[mask].copy(),
        weights_used=counts[mask].copy(),
        all_levels=levels.copy(),
        all_log2_vars=log2_vars.copy(),
        all_counts=counts.copy(),
    )


def estimate_A_given_alpha_vector(
    x: np.ndarray,
    alpha_hat_vec: np.ndarray,
    ridge: float = 1e-8,
) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    alpha_hat_vec = np.asarray(alpha_hat_vec, dtype=float).reshape(-1)
    T = x.shape[0] - 1
    n = x.shape[1]

    if T < 1:
        raise ValueError("Need at least one transition to estimate A.")
    if alpha_hat_vec.shape[0] != n:
        raise ValueError("alpha_hat_vec must have length n.")

    Xprev = x[:T].T
    G = Xprev @ Xprev.T + ridge * np.eye(n)
    A_hat = np.zeros((n, n), dtype=float)

    for i in range(n):
        y_row_full = delta_alpha_prefix_1d(x[:, i], float(alpha_hat_vec[i]))
        y_row = y_row_full[1 : T + 1].reshape(1, T)
        rhs = y_row @ Xprev.T
        #a_hat = np.linalg.solve(G, rhs.T).T
        a_hat = np.linalg.lstsq(G, rhs.T, rcond=None)[0].T 
        A_hat[i, :] = a_hat.reshape(-1)

    return A_hat


@dataclass
class WaveletVectorOLSBaselineResult:
    alpha_hat_vec: np.ndarray
    slope_hat_coords: np.ndarray
    A_hat: np.ndarray
    coord_diagnostics: List[WaveletAlpha1DResult]


def estimate_wavelet_ols_single_trajectory(
    x: np.ndarray,
    alpha_min: float = 0.05,
    alpha_max: float = 0.95,
    ridge: float = 1e-8,
    min_level: int = 2,
    max_level: Optional[int] = None,
    detrend: str = "linear",
    common_alpha_agg: str = "mean",  
) -> WaveletVectorOLSBaselineResult:
    x = np.asarray(x, dtype=float)
    n = x.shape[1]

    diag_results: List[WaveletAlpha1DResult] = []
    alpha_coords = np.zeros(n, dtype=float)
    slope_coords = np.zeros(n, dtype=float)

    for i in range(n):
        res_i = estimate_alpha_wavelet_1d(
            x[:, i],
            alpha_min=alpha_min,
            alpha_max=alpha_max,
            min_level=min_level,
            max_level=max_level,
            detrend=detrend,
        )
        diag_results.append(res_i)
        alpha_coords[i] = res_i.alpha_hat
        slope_coords[i] = res_i.slope_hat

    alpha_coords = np.clip(alpha_coords, alpha_min, alpha_max)
    A_hat = estimate_A_given_alpha_vector(x, alpha_hat_vec=alpha_coords, ridge=ridge)

    return WaveletVectorOLSBaselineResult(
        alpha_hat_vec=alpha_coords,
        slope_hat_coords=slope_coords,
        A_hat=A_hat,
        coord_diagnostics=diag_results,
    )


# ============================================================
# Metrics
# ============================================================
def matrix_entrywise_mse(M_hat: np.ndarray, M_true: np.ndarray) -> float:
    return float(np.mean((np.asarray(M_hat) - np.asarray(M_true)) ** 2))


def vector_mse(v_hat: np.ndarray, v_true: np.ndarray) -> float:
    return float(np.mean((np.asarray(v_hat) - np.asarray(v_true)) ** 2))


def _mean_and_se_cases_trials(arr: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(arr, dtype=float)
    flat = arr.reshape(arr.shape[0], -1)
    mean = np.mean(flat, axis=1)
    if flat.shape[1] > 1:
        se = np.std(flat, axis=1, ddof=1) / np.sqrt(flat.shape[1])
    else:
        se = np.zeros(flat.shape[0], dtype=float)
    return mean, se




# ============================================================
# Comparison experiment
# ============================================================
def run_three_method_comparison_experiment(
    pair_list: List[Dict[str, Any]],
    T_list: np.ndarray,
    N_trials: int,
    sigma_eta: float,
    base_seed: int = 0,
    x0: Optional[np.ndarray] = None,
    x0_mean: float | np.ndarray = 0.0,
    x0_std: float = 1.0,
    fosysid_alpha_bounds: Optional[np.ndarray] = None,
    fosysid_eps_vec: Optional[np.ndarray] = None,
    fosysid_num_grid_vec: Optional[np.ndarray] = None,
    fosysid_ridge: float = 1e-8,
    paper_p: int = 20,
    paper_tol: float = 1e-2,
    paper_max_iter: int = 25,
    paper_ridge: float = 1e-8,
    paper_alpha_left: float = 0.05,
    paper_alpha_right: float = 0.95,
    paper_mse_mode: str = "rollout",
    paper_update_rule: str = "literal_endpoints",
    paper_recover_A_mode: str = "user_minus",
    wavelet_alpha_min: float = 0.05,
    wavelet_alpha_max: float = 0.95,
    wavelet_ridge: float = 1e-8,
    wavelet_min_level: int = 2,
    wavelet_max_level: Optional[int] = None,
    wavelet_detrend: str = "linear",
    wavelet_common_alpha_agg: str = "mean",
    verbose: bool = True,
) -> Dict[str, Any]:
    if len(pair_list) == 0:
        raise ValueError("pair_list must be nonempty.")

    A0 = np.asarray(pair_list[0]["A_true"], dtype=float)
    if A0.shape != (2, 2):
        raise ValueError("This comparison is configured for the n=2 case.")
    n = A0.shape[0]

    normalized_pair_list = []
    for idx, pair in enumerate(pair_list):
        A_chk = np.asarray(pair["A_true"], dtype=float)
        alpha_chk = np.asarray(pair["alpha_true_vec"], dtype=float).reshape(-1)
        eigvals_chk = np.asarray(pair.get("eigvals_true", np.linalg.eigvals(A_chk)))

        if A_chk.shape != (n, n):
            raise ValueError(f"pair_list[{idx}]['A_true'] must have shape ({n}, {n}).")
        if alpha_chk.shape != (n,):
            raise ValueError(f"pair_list[{idx}]['alpha_true_vec'] must have shape ({n},).")

        normalized_pair_list.append(
            {
                "A_true": A_chk,
                "alpha_true_vec": alpha_chk,
                "eigvals_true": eigvals_chk,
            }
        )

    n_cases = len(normalized_pair_list)
    T_list = np.asarray(T_list, dtype=int)

    if fosysid_alpha_bounds is None:
        fosysid_alpha_bounds = np.tile(np.array([[0.05, 0.95]], dtype=float), (n, 1))
    fosysid_alpha_bounds = np.asarray(fosysid_alpha_bounds, dtype=float)
    if fosysid_alpha_bounds.shape != (n, 2):
        raise ValueError(f"fosysid_alpha_bounds must have shape ({n}, 2).")

    # Method 1: FO-GS
    fosysid_alpha_hats = np.zeros((len(T_list), n_cases, N_trials, n), dtype=float)
    fosysid_A_hats = np.zeros((len(T_list), n_cases, N_trials, n, n), dtype=float)
    fosysid_alpha_mse = np.zeros((len(T_list), n_cases, N_trials), dtype=float)
    fosysid_A_mse = np.zeros((len(T_list), n_cases, N_trials), dtype=float)

    # Method 2: FO-BS
    paper_alpha_hats = np.zeros((len(T_list), n_cases, N_trials, n), dtype=float)
    paper_A_hats = np.zeros((len(T_list), n_cases, N_trials, n, n), dtype=float)
    paper_alpha_mse = np.zeros((len(T_list), n_cases, N_trials), dtype=float)
    paper_A_mse = np.zeros((len(T_list), n_cases, N_trials), dtype=float)

    # Method 3: FO-WT
    wavelet_alpha_hats = np.zeros((len(T_list), n_cases, N_trials, n), dtype=float)
    wavelet_A_hats = np.zeros((len(T_list), n_cases, N_trials, n, n), dtype=float)
    wavelet_alpha_mse = np.zeros((len(T_list), n_cases, N_trials), dtype=float)
    wavelet_A_mse = np.zeros((len(T_list), n_cases, N_trials), dtype=float)

    for t_idx, T in enumerate(T_list):
        if verbose:
            print(f"Running T={int(T)} ({t_idx + 1}/{len(T_list)})")

        for case_idx in range(n_cases):
            pair = normalized_pair_list[case_idx]
            A_true = pair["A_true"]
            alpha_true_vec = pair["alpha_true_vec"]

            if verbose:
                print(
                    f"  pair {case_idx + 1}/{n_cases}: "
                    f"alpha = {np.round(alpha_true_vec, 4)}, "
                    f"eigvals = {np.round(np.linalg.eigvals(A_true), 4)}"
                )

            for trial in range(N_trials):
                seed = base_seed + 100000 * t_idx + 1000 * case_idx + trial
                rng = np.random.default_rng(seed)

                x = simulate_vector_fractional_system(
                    A_star=A_true,
                    alpha_star_vec=alpha_true_vec,
                    T=int(T),
                    sigma_eta=float(sigma_eta),
                    rng=rng,
                    x0=x0,
                    x0_mean=x0_mean,
                    x0_std=x0_std,
                )

                # Method 1: FO-GS
                fos_res = estimate_fosysid_single_trajectory(
                    x=x,
                    alpha_bounds=fosysid_alpha_bounds,
                    eps_vec=fosysid_eps_vec,
                    num_vec=fosysid_num_grid_vec,
                    ridge=fosysid_ridge,
                )
                fos_alpha_hat = fos_res["alpha_hat"]
                fos_A_hat = fos_res["A_hat"]
                fosysid_alpha_hats[t_idx, case_idx, trial] = fos_alpha_hat
                fosysid_A_hats[t_idx, case_idx, trial] = fos_A_hat
                fosysid_alpha_mse[t_idx, case_idx, trial] = vector_mse(fos_alpha_hat, alpha_true_vec)
                fosysid_A_mse[t_idx, case_idx, trial] = matrix_entrywise_mse(fos_A_hat, A_true)

                # Method 2: FO-BS
                pap_res = estimate_dt_fods_paper_aligned(
                    x=x,
                    p=paper_p,
                    tol=paper_tol,
                    max_iter=paper_max_iter,
                    ridge=paper_ridge,
                    alpha_left=paper_alpha_left,
                    alpha_right=paper_alpha_right,
                    mse_mode=paper_mse_mode,
                    update_rule=paper_update_rule,
                    recover_A_mode=paper_recover_A_mode,
                )
                pap_alpha_hat = pap_res.alpha_hat
                pap_A_hat = pap_res.A_hat_from_Atilde
                paper_alpha_hats[t_idx, case_idx, trial] = pap_alpha_hat
                paper_A_hats[t_idx, case_idx, trial] = pap_A_hat
                paper_alpha_mse[t_idx, case_idx, trial] = vector_mse(pap_alpha_hat, alpha_true_vec)
                paper_A_mse[t_idx, case_idx, trial] = matrix_entrywise_mse(pap_A_hat, A_true)

                # Method 3: FO-WT
                wav_res = estimate_wavelet_ols_single_trajectory(
                    x=x,
                    alpha_min=wavelet_alpha_min,
                    alpha_max=wavelet_alpha_max,
                    ridge=wavelet_ridge,
                    min_level=wavelet_min_level,
                    max_level=wavelet_max_level,
                    detrend=wavelet_detrend,
                    common_alpha_agg=wavelet_common_alpha_agg,
                )
                wav_alpha_hat = wav_res.alpha_hat_vec
                wav_A_hat = wav_res.A_hat
                wavelet_alpha_hats[t_idx, case_idx, trial] = wav_alpha_hat
                wavelet_A_hats[t_idx, case_idx, trial] = wav_A_hat
                wavelet_alpha_mse[t_idx, case_idx, trial] = vector_mse(wav_alpha_hat, alpha_true_vec)
                wavelet_A_mse[t_idx, case_idx, trial] = matrix_entrywise_mse(wav_A_hat, A_true)

    fos_alpha_mean, fos_alpha_se = _mean_and_se_cases_trials(fosysid_alpha_mse)
    fos_A_mean, fos_A_se = _mean_and_se_cases_trials(fosysid_A_mse)
    pap_alpha_mean, pap_alpha_se = _mean_and_se_cases_trials(paper_alpha_mse)
    pap_A_mean, pap_A_se = _mean_and_se_cases_trials(paper_A_mse)
    wav_alpha_mean, wav_alpha_se = _mean_and_se_cases_trials(wavelet_alpha_mse)
    wav_A_mean, wav_A_se = _mean_and_se_cases_trials(wavelet_A_mse)

    return {
        "pair_list": normalized_pair_list,
        "T_list": T_list,
        "N_trials": int(N_trials),
        "n_pairs": int(n_cases),
        "sigma_eta": float(sigma_eta),
        "fosysid_alpha_hats": fosysid_alpha_hats,
        "fosysid_A_hats": fosysid_A_hats,
        "fosysid_alpha_mse_trials": fosysid_alpha_mse,
        "fosysid_A_mse_trials": fosysid_A_mse,
        "paper_alpha_hats": paper_alpha_hats,
        "paper_A_hats": paper_A_hats,
        "paper_alpha_mse_trials": paper_alpha_mse,
        "paper_A_mse_trials": paper_A_mse,
        "wavelet_alpha_hats": wavelet_alpha_hats,
        "wavelet_A_hats": wavelet_A_hats,
        "wavelet_alpha_mse_trials": wavelet_alpha_mse,
        "wavelet_A_mse_trials": wavelet_A_mse,
        "fosysid_alpha_mse_mean": fos_alpha_mean,
        "fosysid_alpha_mse_se": fos_alpha_se,
        "fosysid_A_mse_mean": fos_A_mean,
        "fosysid_A_mse_se": fos_A_se,
        "paper_alpha_mse_mean": pap_alpha_mean,
        "paper_alpha_mse_se": pap_alpha_se,
        "paper_A_mse_mean": pap_A_mean,
        "paper_A_mse_se": pap_A_se,
        "wavelet_alpha_mse_mean": wav_alpha_mean,
        "wavelet_alpha_mse_se": wav_alpha_se,
        "wavelet_A_mse_mean": wav_A_mean,
        "wavelet_A_mse_se": wav_A_se,
    }

def _set_fixed_y_ticks(ax, top_tick: float, step: float):
    ax.set_ylim(0.0, top_tick)
    ax.set_yticks(np.arange(step, top_tick + 0.5 * step, step))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))

def make_summary_tables(results: Dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame]:
    alpha_df = pd.DataFrame(
        {
            "T": results["T_list"],
            "proposed_alpha_vector_MSE": results["fosysid_alpha_mse_mean"],
            "paper_alpha_vector_MSE": results["paper_alpha_mse_mean"],
            "wavelet_alpha_vector_MSE": results["wavelet_alpha_mse_mean"],
        }
    )
    A_df = pd.DataFrame(
        {
            "T": results["T_list"],
            "proposed_A_MSE": results["fosysid_A_mse_mean"],
            "paper_A_MSE": results["paper_A_mse_mean"],
            "wavelet_A_MSE": results["wavelet_A_mse_mean"],
        }
    )
    return alpha_df, A_df


def _apply_matlab_like_style(ax):
    ax.set_facecolor("white")
    ax.figure.patch.set_facecolor("white")

    for spine in ax.spines.values():
        spine.set_color("0.5")
        spine.set_linewidth(0.8)

    ax.tick_params(
        axis="both",
        which="both",
        direction="in",
        top=True,
        right=True,
        color="0.5",
        labelcolor="black",
        labelsize=18,
    )

    ax.grid(False)


def plot_mse_vs_T_three_methods(
    results: Dict[str, Any],
    save_dir: Optional[str | Path] = None,
    show_ci: bool = True,
    ci_z: float = 1.96,
):
    T = np.asarray(results["T_list"])

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

    methods = [
        ("FO-GS", "fosysid", "red"),
        ("FO-BS", "paper", "blue"),
        ("FO-WT", "wavelet", "#00D7EB"),  
    ]

    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": 18,
        "axes.labelsize": 18,
        "legend.fontsize": 18,
        "xtick.labelsize": 18,
        "ytick.labelsize": 18,
    })

    # -------------------------
    # alpha MSE plot
    # -------------------------
    fig, ax = plt.subplots(figsize=(7, 5.25))
    _apply_matlab_like_style(ax)


    for label, key, color in methods:
        mean = np.asarray(results[f"{key}_alpha_mse_mean"])
        se = np.asarray(results[f"{key}_alpha_mse_se"])

        ax.plot(T, mean, color=color, linewidth=1.6, label=label)

        if show_ci:
            lower = np.maximum(mean - ci_z * se, 0.0)
            upper = mean + ci_z * se
            ax.fill_between(T, lower, upper, color=color, alpha=0.12, linewidth=0)


    ax.set_xlabel(r"trajectory horizon ($t$)")
    ax.set_ylabel(r"MSE of $\boldsymbol{\alpha}$")
    _set_fixed_y_ticks(ax, top_tick=0.06, step=0.01)

    legend = ax.legend(
        loc="best",
        frameon=True,
        fancybox=False,
        framealpha=1.0,
        borderpad=0.35,
        handlelength=1.8,
        handletextpad=0.4,
    )
    legend.get_frame().set_facecolor("white")
    legend.get_frame().set_edgecolor("0.5")
    legend.get_frame().set_linewidth(0.8)

    fig.tight_layout()

    if save_dir is not None:
        fig.savefig(
            save_dir / "alpha_vector_mse_vs_T_three_methods.pdf",
            bbox_inches="tight",
            facecolor=fig.get_facecolor(),
            format="pdf",
        )

    plt.show()

    # -------------------------
    # A MSE plot
    # -------------------------
    fig, ax = plt.subplots(figsize=(7, 5.25))
    _apply_matlab_like_style(ax)

    for label, key, color in methods:
        mean = np.asarray(results[f"{key}_A_mse_mean"])
        se = np.asarray(results[f"{key}_A_mse_se"])

        ax.plot(T, mean, color=color, linewidth=1.6, label=label)

        if show_ci:
            lower = np.maximum(mean - ci_z * se, 0.0)
            upper = mean + ci_z * se
            ax.fill_between(T, lower, upper, color=color, alpha=0.12, linewidth=0)

    ax.set_xlabel(r"trajectory horizon ($t$)")
    ax.set_ylabel(r"MSE of $A$")

    _set_fixed_y_ticks(ax, top_tick=0.08, step=0.02)

    legend = ax.legend(
        loc="best",
        frameon=True,
        fancybox=False,
        framealpha=1.0,
        borderpad=0.35,
        handlelength=1.8,
        handletextpad=0.4,
    )
    legend.get_frame().set_facecolor("white")
    legend.get_frame().set_edgecolor("0.5")
    legend.get_frame().set_linewidth(0.8)

    fig.tight_layout()

    if save_dir is not None:
        fig.savefig(
            save_dir / "A_mse_vs_T_three_methods.pdf",
            format="pdf",
            bbox_inches="tight",
            facecolor=fig.get_facecolor(),
        )

    plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# Run the same experiment over many seeds
# Each seed gets its own folder and saved results
# ============================================================
n = 2
if n != 2:
    raise ValueError("This notebook is configured for n=2.")

base_seed = 13376
num_instance_seeds = 100

instance_seed_list = [base_seed + i for i in range(num_instance_seeds)]

# fixed problem setup
n_pairs = 5
alpha_sampling_low = 0.1
alpha_sampling_high = 0.5

# Horizons to compare
T_list = np.array([50, 100, 150, 200, 300, 400, 500], dtype=int)

# Monte Carlo repetitions per T and per matched pair
N_trials = 20

# Noise and initial condition
sigma_eta = 0.1
x0 = None
x0_mean = 0.0
x0_std = 4.0

# FO-GS
fosysid_alpha_bounds = np.array(
    [
        [0.05, 0.55],
        [0.05, 0.55],
    ],
    dtype=float,
)
fosysid_eps_vec = None
fosysid_num_grid_vec = np.array([20, 20], dtype=int)
fosysid_ridge = 1e-6

# FO-BS
paper_p = 40
paper_tol = 1e-2
paper_max_iter = 100
paper_ridge = 1e-6
paper_alpha_left = 0.05
paper_alpha_right = 0.55
paper_mse_mode = "rollout"
paper_update_rule = "literal_endpoints"
paper_recover_A_mode = "user_minus"

# FO-WT
wavelet_alpha_min = 0.05
wavelet_alpha_max = 0.55
wavelet_ridge = 1e-6
wavelet_min_level = 2
wavelet_max_level = None
wavelet_detrend = "linear"
wavelet_common_alpha_agg = "mean"

base_seed = 3123

root_save_dir = Path("comparison_three_methods_vector_alpha_pairs_many_seeds")
root_save_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []

for seed_idx, instance_seed in enumerate(instance_seed_list, start=1):
    print("=" * 80)
    print(f"Running seed {seed_idx}/{len(instance_seed_list)}: instance_seed = {instance_seed}")
    print("=" * 80)

    instance_rng = np.random.default_rng(instance_seed)

    pair_list = []
    for k in range(n_pairs):
        A_k, eigvals_k = sample_stable_A(
            n=n,
            rng=instance_rng,
            eig_low=-0.5,
            eig_high=0.5,
            symmetric=False,
        )

        alpha_k = instance_rng.uniform(
            alpha_sampling_low,
            alpha_sampling_high,
            size=n,
        )

        pair_list.append(
            {
                "A_true": A_k,
                "eigvals_true": eigvals_k,
                "alpha_true_vec": alpha_k,
            }
        )

    for k, pair in enumerate(pair_list, start=1):
        print(f"Pair {k}:")
        print("A_true =")
        print(pair["A_true"])
        print("sampled eigenvalues =", np.round(pair["eigvals_true"], 6))
        print("alpha_true_vec =", np.round(pair["alpha_true_vec"], 6))
        print()

    save_dir = root_save_dir / f"seed_{instance_seed}"
    save_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # run experiment
    # --------------------------------------------------------
    results = run_three_method_comparison_experiment(
        pair_list=pair_list,
        T_list=T_list,
        N_trials=N_trials,
        sigma_eta=sigma_eta,
        base_seed=base_seed,
        x0=x0,
        x0_mean=x0_mean,
        x0_std=x0_std,
        fosysid_alpha_bounds=fosysid_alpha_bounds,
        fosysid_eps_vec=fosysid_eps_vec,
        fosysid_num_grid_vec=fosysid_num_grid_vec,
        fosysid_ridge=fosysid_ridge,
        paper_p=paper_p,
        paper_tol=paper_tol,
        paper_max_iter=paper_max_iter,
        paper_ridge=paper_ridge,
        paper_alpha_left=paper_alpha_left,
        paper_alpha_right=paper_alpha_right,
        paper_mse_mode=paper_mse_mode,
        paper_update_rule=paper_update_rule,
        paper_recover_A_mode=paper_recover_A_mode,
        wavelet_alpha_min=wavelet_alpha_min,
        wavelet_alpha_max=wavelet_alpha_max,
        wavelet_ridge=wavelet_ridge,
        wavelet_min_level=wavelet_min_level,
        wavelet_max_level=wavelet_max_level,
        wavelet_detrend=wavelet_detrend,
        wavelet_common_alpha_agg=wavelet_common_alpha_agg,
        verbose=True,
    )

    # --------------------------------------------------------
    # summaries
    # --------------------------------------------------------
    alpha_df, A_df = make_summary_tables(results)

    print("Alpha-vector MSE summary")
    print(alpha_df)
    print("\nA MSE summary")
    print(A_df)

    alpha_df.to_csv(save_dir / "alpha_summary.csv", index=False)
    A_df.to_csv(save_dir / "A_summary.csv", index=False)

    plot_mse_vs_T_three_methods(results, save_dir=save_dir, show_ci=True)
    print(f"Saved figures to: {save_dir.resolve()}")

    pair_A_true = np.stack([pair["A_true"] for pair in results["pair_list"]], axis=0)
    pair_alpha_true = np.stack([pair["alpha_true_vec"] for pair in results["pair_list"]], axis=0)
    pair_eigvals_true = np.stack([pair["eigvals_true"] for pair in results["pair_list"]], axis=0)

    np.savez(
        save_dir / "vector_alpha_three_method_results.npz",
        pair_A_true=pair_A_true,
        pair_alpha_true=pair_alpha_true,
        pair_eigvals_true=pair_eigvals_true,
        T_list=results["T_list"],
        fosysid_alpha_hats=results["fosysid_alpha_hats"],
        fosysid_A_hats=results["fosysid_A_hats"],
        paper_alpha_hats=results["paper_alpha_hats"],
        paper_A_hats=results["paper_A_hats"],
        wavelet_alpha_hats=results["wavelet_alpha_hats"],
        wavelet_A_hats=results["wavelet_A_hats"],
        fosysid_alpha_mse_trials=results["fosysid_alpha_mse_trials"],
        fosysid_A_mse_trials=results["fosysid_A_mse_trials"],
        paper_alpha_mse_trials=results["paper_alpha_mse_trials"],
        paper_A_mse_trials=results["paper_A_mse_trials"],
        wavelet_alpha_mse_trials=results["wavelet_alpha_mse_trials"],
        wavelet_A_mse_trials=results["wavelet_A_mse_trials"],
    )

    summary_rows.append(
        {
            "instance_seed": instance_seed,
            "save_dir": str(save_dir),
            "fosysid_alpha_mean_over_T": float(np.mean(results["fosysid_alpha_mse_mean"])),
            "fosysid_A_mean_over_T": float(np.mean(results["fosysid_A_mse_mean"])),
            "paper_alpha_mean_over_T": float(np.mean(results["paper_alpha_mse_mean"])),
            "paper_A_mean_over_T": float(np.mean(results["paper_A_mse_mean"])),
            "wavelet_alpha_mean_over_T": float(np.mean(results["wavelet_alpha_mse_mean"])),
            "wavelet_A_mean_over_T": float(np.mean(results["wavelet_A_mse_mean"])),
        }
    )


summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(root_save_dir / "seed_summary.csv", index=False)

print("\nFinished all seeds.")
print(summary_df)
print(f"\nSeed summary saved to: {(root_save_dir / 'seed_summary.csv').resolve()}")